# Final solution: industrial equipment failure prediction

This notebook is reproducible from the three competition files in the repository root:

- `train.csv`
- `test.csv`
- `sample_submission.csv`

It runs sklearn grid-search experiments with ROC AUC, saves experiment tables and plots to `predictive_maintenance/experiments/`, trains the best selected pipeline on all training rows, and writes `fin_submission.csv` with probabilities in `Machine failure`.


## Approach

The target is highly imbalanced, so all model selection uses stratified cross-validation and `roc_auc`. The workflow tests:

1. Numeric/categorical preprocessing with different `SimpleImputer` strategies.
2. Domain feature engineering, including the mechanical-power formula:

   `Power = 2 * pi * Torque * Rotational_speed / 60`

3. Several sklearn model families: logistic regression, histogram gradient boosting, random forest, and extra trees.
4. Two feature sets: sensor/engineered features only, and sensor/engineered features plus the failure-mode flag columns present in the CSV files.


In [ ]:
from pathlib import Path

import pandas as pd

from predictive_maintenance.solution import ExperimentConfig, read_data, run_experiment


In [ ]:
config = ExperimentConfig(
    train_path=Path('train.csv'),
    test_path=Path('test.csv'),
    sample_submission_path=Path('sample_submission.csv'),
    output_dir=Path('predictive_maintenance/experiments'),
    cv_splits=5,
    quick=False,
)

train, test, sample_submission = read_data(config)
print('train:', train.shape)
print('test:', test.shape)
print('sample_submission:', sample_submission.shape)
print('positive class rate:', train['Machine failure'].mean())
train.head()


## Run experiments and create submission

The next cell performs the grid searches and writes all artifacts to `predictive_maintenance/experiments/`. It also writes `fin_submission.csv` in the repository root. If you only want a fast smoke test, temporarily set `quick=True` in `ExperimentConfig`; for the competition submission keep `quick=False`.


In [ ]:
result = run_experiment(config)
result


## Inspect experiment summary


In [ ]:
summary_path = config.output_dir / 'experiment_summary.csv'
summary = pd.read_csv(summary_path)
summary[['model_name', 'include_failure_mode_flags', 'best_oof_auc', 'mean_test_score', 'std_test_score', 'params']].head(10)


## Validate final submission format


In [ ]:
submission = pd.read_csv('fin_submission.csv')
assert list(submission.columns) == ['id', 'Machine failure']
assert len(submission) == len(sample_submission)
assert submission['Machine failure'].between(0, 1).all()
submission.head()
